In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


In [ ]:
import numpy as np
import pandas as pd
from data.generation import improve_resolution, add_cloud, gen_case, gen_load, gen_pv, gen_measure
from configs.config import sys_config, opt
from pypower.api import runopf
from physics.state_estimation import SE
from configs.config_mea_idx import define_mea_idx_noise
import matplotlib.pyplot as plt

from pathlib import Path
Path('data', sys_config['case_name']).mkdir(parents=True, exist_ok=True)
print(opt)
print(sys_config)

In [ ]:
# Load raw data
raw_load_dir = 'data/raw_data/load.csv'
raw_pv_dir = 'data/raw_data/pv.csv'
raw_load = pd.read_csv(raw_load_dir)
raw_pv = pd.read_csv(raw_pv_dir)
raw_load = raw_load[(raw_load['DateTime'] <= '2012-09-30 23:45:00') & (raw_load['DateTime'] >= '2012-06-00 00:00:00')]
raw_pv = raw_pv[(raw_pv['DateTime'] <= '2019-09-30 23:45:00') & (raw_pv['DateTime'] >= '2019-06-00 00:00:00')]
raw_load.reset_index(inplace=True, drop=True)
raw_pv.reset_index(inplace=True, drop = True)

In [ ]:
# Improve data resolution
high_load_dir = 'data/raw_data/load_high.csv'
high_pv_dir = 'data/raw_data/pv_high.csv'
load_high, pv_high = improve_resolution(load_raw = raw_load, pv_raw = raw_pv, res = '5min')

# Add cloud (large deviations) to the pv dat
pv_new = add_cloud(pv_high, unchange_rate=0.5, max_reduce=0.8)

# Save
load_high.to_csv(high_load_dir, index_label='DateTime')
pv_high.to_csv(high_pv_dir, index_label='DateTime')

In [ ]:
# Modify case
case = gen_case(sys_config['case_name'])
# case = gen_case('case14')
print(f'Default active load: {case["bus"][:,2]}')
print(f'Default reactive load: {case["bus"][:,3]}')
print(f'Default total active load: {case["bus"][:,2].sum()}')
print(f'Default generation limit: {case["gen"][:,8].sum()}')
# Test OPF on it
result = runopf(case, opt)
print(f'Success: {result["success"]}')
print(f'Voltage phase angle: {result["bus"][:,8]}')

In [ ]:
# Generate load for case
high_load_dir = 'data/raw_data/load_high.csv'
high_pv_dir = 'data/raw_data/pv_high.csv'

load_high = pd.read_csv(high_load_dir)
pv_high = pd.read_csv(high_pv_dir)

load_active_dir = f'data/{sys_config["case_name"]}/load_active.npy'
load_reactive_dir = f'data/{sys_config["case_name"]}/load_reactive.npy'
load_active, load_reactive = gen_load(case = case, load_raw = load_high)
np.save(load_active_dir, load_active, allow_pickle=True)
np.save(load_reactive_dir, load_reactive, allow_pickle=True)

pv_active_dir = f'data/{sys_config["case_name"]}/pv_active.npy'
pv_reactive_dir = f'data/{sys_config["case_name"]}/pv_reactive.npy'
pv_active, pv_reactive = gen_pv(sys_config['pv_bus'], pv_high, load_active, penetration_ratio = 0.3)
np.save(pv_active_dir, pv_active, allow_pickle=True)
np.save(pv_reactive_dir, pv_reactive, allow_pickle=True)

In [ ]:
# Test if the maximum loads can converge in OPF
case = gen_case(sys_config['case_name'])
load_active_dir = f'data/{sys_config["case_name"]}/load_active.npy'
load_reactive_dir = f'data/{sys_config["case_name"]}/load_reactive.npy'

pv_active_dir = f'data/{sys_config["case_name"]}/pv_active.npy'
pv_reactive_dir = f'data/{sys_config["case_name"]}/pv_reactive.npy'

noise_sigma_dir = f'data/{sys_config["case_name"]}/noise_sigma.npy'

load_active = np.load(load_active_dir)
load_reactive = np.load(load_reactive_dir)

pv_active = np.load(pv_active_dir)
pv_reactive = np.load(pv_reactive_dir)

# Analysis
load_max = np.max(np.sum(load_active, axis = 1))
print(f'Max active load: {load_max}')
pv_max = np.max(np.sum(pv_active, axis = 1))
print(f'Max pv load: {pv_max}')
print(f'PV penetration: {pv_max/load_max}')

# Generate noise sigma
idx, no_mea, noise_sigma = define_mea_idx_noise(case, choice = sys_config['measure_type'])
case_class = SE(case, noise_sigma, idx, fpr = sys_config['fpr'])
result = case_class.run_opf(verbose=False)

print("opf success:", result['success'])

print("Vm min:", np.min(result['bus'][:,7]))
print("Vm max:", np.max(result['bus'][:,7]))

print("Va min:", np.min(result['bus'][:,8]))
print("Va max:", np.max(result['bus'][:,8]))


z, z_noise, vang_ref, vmag_ref = case_class.construct_mea(result)
print(f'z shape: {z_noise.shape}')
noise_sigma = np.abs(z * sys_config['noise_ratio_power']).flatten() + 1e-3
np.save(noise_sigma_dir, noise_sigma, allow_pickle=True)
plt.plot(noise_sigma)

In [ ]:
noise_sigma_dir = f'data/{sys_config["case_name"]}/noise_sigma.npy'
case = gen_case(sys_config['case_name'])
idx, no_mea, noise_sigma = define_mea_idx_noise(case, choice = sys_config['measure_type'])
noise_sigma = np.load(noise_sigma_dir)
case_class = SE(case, noise_sigma, idx, fpr = sys_config['fpr'])

load_active_dir = f'data/{sys_config["case_name"]}/load_active.npy'
load_reactive_dir = f'data/{sys_config["case_name"]}/load_reactive.npy'
pv_active_dir = f'data/{sys_config["case_name"]}/pv_active.npy'
pv_reactive_dir = f'data/{sys_config["case_name"]}/pv_reactive.npy'

load_active = np.load(load_active_dir)
load_reactive = np.load(load_reactive_dir)
pv_active = np.load(pv_active_dir)
pv_reactive = np.load(pv_reactive_dir)

pv_active_ = np.zeros((load_active.shape[0], load_reactive.shape[1]))
pv_reactive_ = np.zeros((load_reactive.shape[0], load_reactive.shape[1]))
pv_active_[:,sys_config['pv_bus']] = pv_active
pv_reactive_[:,sys_config['pv_bus']] = pv_reactive

largest_load_idx = np.flip(np.argsort(np.sum(load_active - pv_active_, axis = 1)))[:10]

for i in range(10):
    result = case_class.run_opf(verbose=False, load_active = load_active[i] - pv_active_[i], load_reactive = load_reactive[i])
    print(result['success'])
    print(f'Voltage angle: {result["bus"][:,8]*np.pi/180}')
    break

"""
To make sure the largest load case can converge
"""

for i in largest_load_idx:
    result = case_class.run_opf(verbose=False, load_active = load_active[i] - pv_active_[i], load_reactive = load_reactive[i])
    print(f'total load: {np.sum(load_active[i] - pv_active_[i])}')
    print(result['success'])